# Credit Risk ETL Practice Project

This project is a small Python practice workflow for credit risk reporting.

It creates fake loan-level data, performs basic data cleaning, calculates expected loss using PD, LGD, and EAD, applies a simple stress scenario, and summarizes results by portfolio.

## 1. Create fake loan-level data

In [13]:
import pandas as pd
import numpy as np

# Set random seed so results are repeatable
np.random.seed(42)

# Creating a simple fake loan data
n = 100

loans = pd.DataFrame({
    "loan_id": [f"L{i:04d}" for i in range(1, n + 1)],
    "borrower_id": [f"B{i:04d}" for i in np.random.randint(1, 80, size=n)],
    "portfolio": np.random.choice(
        ["Mortgage", "Auto Loan", "Credit Card", "Personal Loan"],
        size=n,
        p=[0.45, 0.20, 0.20, 0.15]
    ),
    "region": np.random.choice(
        ["Ontario", "Quebec", "British Columbia", "Alberta"],
        size=n
    ),
    "pd": np.round(np.random.uniform(0.001, 0.10, size=n), 4),
    "lgd": np.round(np.random.uniform(0.20, 0.60, size=n), 4),
    "ead": np.random.randint(10_000, 1_000_000, size=n),
    "as_of_date": "2026-05-22"
})

# Add one duplicate row on purpose, for practicing cleaning
loans = pd.concat([loans, loans.iloc[[0]]], ignore_index=True)

# Add one missing value on purpose
loans.loc[5, "lgd"] = np.nan

# Save raw fake data
loans.to_csv("fake_loans.csv", index=False)

loans.head()

,loan_id,borrower_id,portfolio,region,pd,lgd,ead,as_of_date
0,L0001,B0052,Mortgage,Quebec,0.0649,0.2481,223945,2026-05-22
1,L0002,B0015,Mortgage,British Columbia,0.0672,0.3368,241915,2026-05-22
2,L0003,B0072,Mortgage,British Columbia,0.0866,0.2367,737952,2026-05-22
3,L0004,B0061,Credit Card,Quebec,0.0238,0.2377,218124,2026-05-22
4,L0005,B0021,Credit Card,Quebec,0.0504,0.3246,447477,2026-05-22


## 2. Inspect raw data

In [14]:
df = pd.read_csv("fake_loans.csv")

# Checking and looking at first few rows
display(df.head())

# Basic structure: columns, data types, non-null counts
df.info()

# Summary statistics for numeric columns
display(df.describe())

# Check missing values by column
display(df.isna().sum())

# Check number of duplicate rows
print("Number of duplicate rows:", df.duplicated().sum())

,loan_id,borrower_id,portfolio,region,pd,lgd,ead,as_of_date
0,L0001,B0052,Mortgage,Quebec,0.0649,0.2481,223945,2026-05-22
1,L0002,B0015,Mortgage,British Columbia,0.0672,0.3368,241915,2026-05-22
2,L0003,B0072,Mortgage,British Columbia,0.0866,0.2367,737952,2026-05-22
3,L0004,B0061,Credit Card,Quebec,0.0238,0.2377,218124,2026-05-22
4,L0005,B0021,Credit Card,Quebec,0.0504,0.3246,447477,2026-05-22


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   loan_id      101 non-null    object 
 1   borrower_id  101 non-null    object 
 2   portfolio    101 non-null    object 
 3   region       101 non-null    object 
 4   pd           101 non-null    float64
 5   lgd          100 non-null    float64
 6   ead          101 non-null    int64  
 7   as_of_date   101 non-null    object 
dtypes: float64(2), int64(1), object(5)
memory usage: 6.4+ KB


,pd,lgd,ead
count,101.000000,100.000000,101.000000
mean,0.046802,0.405865,477206.930693
std,0.028747,0.117312,271210.344695
min,0.001500,0.206100,23807.000000
25%,0.023800,0.304375,244677.000000
50%,0.043900,0.418800,470857.000000
75%,0.069200,0.495100,728216.000000
max,0.099500,0.598400,985358.000000


,0
loan_id,0
borrower_id,0
portfolio,0
region,0
pd,0
lgd,1
ead,0
as_of_date,0


Number of duplicate rows: 1


## 3. Clean data

In [15]:
# Keep original row count for comparison
original_rows = len(df)

# Remove duplicate rows
df_clean = df.drop_duplicates()

# Remove rows where important risk fields are missing
df_clean = df_clean.dropna(subset=["pd", "lgd", "ead"])

# Keep only valid risk values and make a separate table
df_clean = df_clean[
    (df_clean["pd"] >= 0) & (df_clean["pd"] <= 1) &
    (df_clean["lgd"] >= 0) & (df_clean["lgd"] <= 1) &
    (df_clean["ead"] > 0)
].copy()

clean_rows = len(df_clean)

print("Original rows:", original_rows)
print("Clean rows:", clean_rows)
print("Rows removed:", original_rows - clean_rows)

display(df_clean.head())

Original rows: 101
Clean rows: 99
Rows removed: 2


,loan_id,borrower_id,portfolio,region,pd,lgd,ead,as_of_date
0,L0001,B0052,Mortgage,Quebec,0.0649,0.2481,223945,2026-05-22
1,L0002,B0015,Mortgage,British Columbia,0.0672,0.3368,241915,2026-05-22
2,L0003,B0072,Mortgage,British Columbia,0.0866,0.2367,737952,2026-05-22
3,L0004,B0061,Credit Card,Quebec,0.0238,0.2377,218124,2026-05-22
4,L0005,B0021,Credit Card,Quebec,0.0504,0.3246,447477,2026-05-22


## 4. Calculate expected loss and stressed expected loss

In [16]:
# Calculate expected loss
df_clean["expected_loss"] = df_clean["pd"] * df_clean["lgd"] * df_clean["ead"]

# Apply a simple stress scenario: PD increases by 20%
df_clean["stressed_pd"] = (df_clean["pd"] * 1.20).clip(upper=1)

# Calculate stressed expected loss
df_clean["stressed_expected_loss"] = (
    df_clean["stressed_pd"] * df_clean["lgd"] * df_clean["ead"]
)

# Calculate the increase in expected loss under stress
df_clean["incremental_loss"] = (
    df_clean["stressed_expected_loss"] - df_clean["expected_loss"]
)

display(df_clean.head())

,loan_id,borrower_id,portfolio,region,pd,lgd,ead,as_of_date,expected_loss,stressed_pd,stressed_expected_loss,incremental_loss
0,L0001,B0052,Mortgage,Quebec,0.0649,0.2481,223945,2026-05-22,3605.892967,0.07788,4327.071560,721.178593
1,L0002,B0015,Mortgage,British Columbia,0.0672,0.3368,241915,2026-05-22,5475.252518,0.08064,6570.303022,1095.050504
2,L0003,B0072,Mortgage,British Columbia,0.0866,0.2367,737952,2026-05-22,15126.702445,0.10392,18152.042935,3025.340489
3,L0004,B0061,Credit Card,Quebec,0.0238,0.2377,218124,2026-05-22,1233.984180,0.02856,1480.781016,246.796836
4,L0005,B0021,Credit Card,Quebec,0.0504,0.3246,447477,2026-05-22,7320.652124,0.06048,8784.782548,1464.130425


## 5. Summarize by portfolio

In [17]:
# Summarize risk metrics by portfolio
portfolio_summary = (
    df_clean
    .groupby("portfolio", as_index=False)
    .agg(
        number_of_loans=("loan_id", "count"),
        total_ead=("ead", "sum"),
        average_pd=("pd", "mean"),
        average_lgd=("lgd", "mean"),
        base_expected_loss=("expected_loss", "sum"),
        stressed_expected_loss=("stressed_expected_loss", "sum"),
        incremental_loss=("incremental_loss", "sum")
    )
)

# Sort by stressed expected loss from highest to lowest
portfolio_summary = portfolio_summary.sort_values(
    by="stressed_expected_loss",
    ascending=False
)

display(portfolio_summary)

# Export summary output
portfolio_summary.to_csv("portfolio_risk_summary.csv", index=False)

,portfolio,number_of_loans,total_ead,average_pd,average_lgd,base_expected_loss,stressed_expected_loss,incremental_loss
2,Mortgage,49,22862789,0.052288,0.408894,442327.471449,530792.965739,88465.494290
1,Credit Card,20,10594947,0.041230,0.388895,148760.124916,178512.149899,29752.024983
0,Auto Loan,19,7040389,0.041021,0.434342,124026.705194,148832.046233,24805.341039
3,Personal Loan,11,6940000,0.039855,0.388382,95282.216279,114338.659535,19056.443256


## 6. Run validation checks

In [18]:
# Simple validation checks

assert df_clean["pd"].between(0, 1).all(), "PD has invalid values"
assert df_clean["lgd"].between(0, 1).all(), "LGD has invalid values"
assert (df_clean["ead"] > 0).all(), "EAD has invalid values"
assert (df_clean["expected_loss"] >= 0).all(), "Expected loss cannot be negative"

print("All validation checks passed.")

All validation checks passed.
